In [ ]:
#@title 🌸 คลิกปุ่ม Play ด้านซ้ายเพื่อเริ่มระบบวาดภาพอนิเมะ (Animagine XL 4.0 Opt)
# 1. ติดตั้งไลบรารีที่จำเป็น
!pip install -q diffusers transformers accelerate torch gradio safetensors

import random
import gradio as gr
import torch
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

# 2. โหลดโมเดล Animagine XL 4.0 Opt
model_id = "cagliostrolab/animagine-xl-4.0-opt"

pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
)

# ตั้งค่า Scheduler ให้เหมาะกับสไตล์อนิเมะ
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipe.scheduler.config
)

# ระบบประหยัดแรม (VRAM) สำหรับ T4 GPU ฟรีของ Colab
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

# 3. ฟังก์ชันสร้างภาพ
def generate_anime_v4(prompt, negative_prompt, aspect_ratio, steps, cfg_scale, seed):
    if aspect_ratio == "จัตุรัส (1:1)":
        width, height = 1024, 1024
    elif aspect_ratio == "แนวตั้ง (9:16 - มือถือ)":
        width, height = 832, 1216
    elif aspect_ratio == "แนวนอน (16:9 - หน้าจอคอม)":
        width, height = 1216, 832
    else:
        width, height = 896, 1152

    if seed == -1 or seed is None:
        actual_seed = random.randint(0, 2147483647)
    else:
        actual_seed = int(seed)

    generator = torch.Generator("cpu").manual_seed(actual_seed)

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=steps,
        guidance_scale=cfg_scale,
        generator=generator,
    ).images[0]

    return image, actual_seed

# 4. สร้างหน้าเว็บ UI ด้วย Gradio
with gr.Blocks() as demo:
    gr.Markdown("# 🌸 AI วาดภาพอนิเมะ (Animagine XL 4.0 Opt)")

    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Prompt (คำสั่งวาดภาพ)",
                lines=3,
                value="1girl, silver hair, blue eyes, stylish casual clothes, sunset city background, masterpiece, high score, great score, absurdres",
            )
            neg_prompt_input = gr.Textbox(
                label="Negative Prompt (สิ่งที่ไม่ต้องการ)",
                lines=2,
                value="lowres, bad anatomy, bad hands, text, error, missing finger, extra digits, fewer digits, cropped, worst quality, low quality, low score, bad score, average score, signature, watermark, username, blurry",
            )
            aspect_dropdown = gr.Dropdown(
                choices=["จัตุรัส (1:1)", "แนวตั้งมาตรฐาน (3:4)", "แนวตั้ง (9:16 - มือถือ)", "แนวนอน (16:9 - หน้าจอคอม)"],
                value="แนวตั้งมาตรฐาน (3:4)",
                label="สัดส่วนภาพ (Aspect Ratio)",
            )
            with gr.Row():
                steps_slider = gr.Slider(minimum=15, maximum=40, value=28, step=1, label="Steps (แนะนำ 25-30)")
                cfg_slider = gr.Slider(minimum=1.0, maximum=10.0, value=5.0, step=0.5, label="CFG Scale (รุ่น 4.0 แนะนำ 4.0 - 5.0)")
            seed_input = gr.Number(value=-1, label="Seed (-1 = สุ่มใหม่ทุกครั้ง)", precision=0)
            generate_btn = gr.Button("✨ วาดภาพอนิเมะ", variant="primary")

        with gr.Column():
            output_image = gr.Image(label="ผลลัพธ์")
            used_seed_output = gr.Number(label="Seed ที่ใช้สร้างภาพนี้ (คัดลอกไปใช้ซ้ำได้)", interactive=False)

    generate_btn.click(
        fn=generate_anime_v4,
        inputs=[prompt_input, neg_prompt_input, aspect_dropdown, steps_slider, cfg_slider, seed_input],
        outputs=[output_image, used_seed_output],
    )

demo.launch(share=True, theme=gr.themes.Soft())